In [1]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hashlib
# %matplotlib qt
# %matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames


In [3]:

def get_and_cache_analytics(analytics_name, session_names, columns=None, force_reload=False):
    hashname = hashlib.md5(f"{analytics_name}-{session_names}-{columns}".encode()).hexdigest()
    os.makedirs("./cache/analytics_cache", exist_ok=True)
    if force_reload or not os.path.exists(f"./cache/analytics_cache/{hashname}.pkl"):
        result = analytics.get_analytics(analytics_name, session_names=session_names, columns=columns)
        with open(f"./cache/analytics_cache/{hashname}.pkl", "w") as f:
            result.to_csv(f)
    else:
        with open(f"./cache/analytics_cache/{hashname}.pkl", "r") as f:
            result = pd.read_csv(f)
    return result

In [4]:
os.getcwd()

'/home/amitsant2000/ethz/VirtualReality/analysisVR/scripted_plotting'

In [5]:
output_dir = "./outputs/experimental/"
data = {}
# nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="DEBUG")



In [6]:
# ephys
# paradigm_ids = [0,1100]
# animal_ids = [10]
# session_ids = [0,1,2,3]

animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

# width = 1400
# height = 1400
# group_by = None

# session_modality_from_nas

In [7]:
session_dirs = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)[0]
for sd in session_dirs:
    print(sd)
session_names = fullfnames2snames(session_dirs)
for sn in session_names:
    print(sn)

# session_modality_from_nas(session_dirs[0], 'facecam_frames', stop=20)
# df = session_modality_from_nas(session_dirs[0], 'TrackKinematics')
# df

2026-02-11 19:04:26,742|DEBUG|618360|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-11 19:04:39,519|DEBUG|618360|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 34 sessions.
2026-02-11 19:04:39,522|DEBUG|618360|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: [1100], animal_ids: [6], session_ids: None, from_date: None, to_date: None
	Merging 34 sessions



/mnt/z/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/mnt/z/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min/2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5
/mnt/z/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-15_15-48_rYL006_P1100_LinearTrackStop_35min/2024-11-15_15-48_rYL006_P1100_LinearTrackStop_35min.hdf5
/mnt/z/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-20_17-46_rYL006_P1100_LinearTrackStop_22min/2024-11-20_17-46_rYL006_P1100_LinearTrackStop_22min.hdf5
/mnt/z/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-21_17-22_rYL006_P1100_LinearTrackStop_25min/2024-11-21_17-22_rYL006_P1100_LinearTrackStop_25min.hdf5
/mnt/z/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-22_

In [8]:
fr_raw = get_and_cache_analytics('FiringRate40msHz', session_names=session_names)


2026-02-11 19:04:39,548|DEBUG|618360|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-11 19:05:04,288|DEBUG|618360|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 34 sessions

2026-02-11 19:05:04,291|DEBUG|618360|analytics|get_analytics
	Processing FiringRate40msHz, (6, 1100, 0) 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/mnt/z/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-11 19:05:05,147|INFO|618360|analytics|get_analytics
	Analytic `FiringRate40msHz` not does not exist for (6, 1100, 0), compute first, or check for typo
2026-02-11 19:05:05,148|DEBUG|618360|analytics|get_analytics
	Processing FiringRate40msHz, (6, 1100, 1) 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5
/mnt/z/BMI/VirtualReality/SpatialSequenceLearning

In [9]:
fr = fr_raw.drop("from_ephys_timestamp", axis=1)
fr.index = fr.index.droplevel((0,1,3, ))
fr.set_index('to_ephys_timestamp', append=True, inplace=True)
fr['global_t'] = np.arange(40_000, 40_000*len(fr)+1, 40_000)
fr.set_index('global_t', append=True, inplace=True)
fr.index = fr.index.rename(['session_id', 'session_t', 'global_t'])
fr

Unit0001  Unit0002  Unit0003  Unit0004  \
session_id session_t  global_t                                              
1          40000      40000               0         0         0         0   
           80000      80000               0         0         0         0   
           120000     120000              0         0         0         0   
           160000     160000              0         0         0         0   
           200000     200000              0         0         0         0   
...                                     ...       ...       ...       ...   
33         4435960000 53320480000         0         0         0         0   
           4436000000 53320520000         0         0         0         0   
           4436040000 53320560000         0         0         0         0   
           4436080000 53320600000         0         0         0         0   
           4436120000 53320640000         0         0         0         0   

                                   Unit0005  Unit0006  Unit0007  Unit0008  \
session_id session_t  global_t                                              
1          40000      40000               0         0         0         0   
           80000      80000               0         0         0         0   
           120000     120000              0         0         0         0   
           160000     160000              0         0         0         0   
           200000     200000              0         0         0         0   
...                                     ...       ...       ...       ...   
33         4435960000 53320480000         0         0         0         0   
           4436000000 53320520000         0         0         0         0   
           4436040000 53320560000         0         0         0         0   
           4436080000 53320600000         0         0         0         0   
           4436120000 53320640000         0         0         0         0   

                                   Unit0009  Unit0010  ...  Unit0068  \
session_id session_t  global_t                         ...             
1          40000      40000               0         0  ...         0   
           80000      80000               0         0  ...         0   
           120000     120000              0         0  ...         0   
           160000     160000              0         0  ...         0   
           200000     200000              0         0  ...         0   
...                                     ...       ...  ...       ...   
33         4435960000 53320480000         0         0  ...        25   
           4436000000 53320520000         0         0  ...        25   
           4436040000 53320560000         0         0  ...        25   
           4436080000 53320600000         0         0  ...        25   
           4436120000 53320640000         0         0  ...         0   

                                   Unit0069  Unit0070  Unit0071  Unit0072  \
session_id session_t  global_t                                              
1          40000      40000               0         0         0         0   
           80000      80000               0         0         0         0   
           120000     120000              0         0         0         0   
           160000     160000              0         0         0         0   
           200000     200000              0         0         0         0   
...                                     ...       ...       ...       ...   
33         4435960000 53320480000         0         0        25         0   
           4436000000 53320520000         0         0        25         0   
           4436040000 53320560000        25         0        25         0   
           4436080000 53320600000         0         0         0         0   
           4436120000 53320640000         0        25         0         0   

                                   Unit0073  Unit0074  Unit0075  Unit0076  \
session_id sess

In [22]:
behavior_glm_input = get_and_cache_analytics('Behavior40msAligned', session_names=session_names)

In [23]:
behavior_glm_input.columns

Index(['paradigm_id', 'animal_id', 'session_id', 'entry_id', 'trial_id', 'cue',
       'trial_outcome', 'choice_R1', 'choice_R2', 'trial_start_pc_timestamp',
       'from_ephys_timestamp', 'to_ephys_timestamp', 'frame_velocity',
       'frame_raw', 'frame_raw_quantile', 'frame_acceleration',
       'abs_frame_acceleration', 'frame_positive_acceleration',
       'frame_negative_acceleration', 'frame_yaw', 'frame_yaw_left',
       'frame_yaw_right', 'frame_pitch', 'frame_pitch_left',
       'frame_pitch_right', 'lick_count', 'post_lick', 'post_reward_sound',
       'post_reward', 'frame_position', 'visible_cue', 'zone_before_reward1',
       'zone_before_reward2', 'zone_between_cues', 'zone_cue2',
       'zone_cue2_passed', 'zone_cue2_visible', 'zone_post_reward',
       'zone_reward1', 'zone_reward2', 'head_angle', 'head_angle_left',
       'head_angle_right', 'head_angle_velocity', 'head_angle_velocity_left',
       'head_angle_velocity_right'],
      dtype='object')

In [24]:
behavior_glm_input_flat = behavior_glm_input.reset_index()
behavior_glm_input_flat

,index,paradigm_id,animal_id,session_id,entry_id,trial_id,cue,trial_outcome,choice_R1,choice_R2,...,zone_cue2_visible,zone_post_reward,zone_reward1,zone_reward2,head_angle,head_angle_left,head_angle_right,head_angle_velocity,head_angle_velocity_left,head_angle_velocity_right
0,0,6,1100,1,0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,1,6,1100,1,1,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2,6,1100,1,2,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,3,6,1100,1,3,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,4,6,1100,1,4,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1333011,1333011,6,1100,33,110898,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1333012,1333012,6,1100,33,110899,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1333013,1333013,6,1100,33,110900,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1333014,1333014,6,1100,33,110901,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
chosen_colums = [
    'session_id',
    'from_ephys_timestamp',
    'to_ephys_timestamp',
    'frame_velocity',
    'frame_acceleration',
    'frame_yaw',
    'frame_raw',
    'frame_pitch',
    'lick_count',
    'frame_position',
    'zone_post_reward',
    'post_reward',
    'post_reward_sound',
    'zone_before_reward1',
    'zone_before_reward2',
    'zone_cue2_passed',
    'head_angle',
    'head_angle_left',
    'head_angle_right',
    'head_angle_velocity',
    'head_angle_velocity_left',
    'head_angle_velocity_right',
]
behavior_glm_input_flat.columns


Index(['index', 'paradigm_id', 'animal_id', 'session_id', 'entry_id',
       'trial_id', 'cue', 'trial_outcome', 'choice_R1', 'choice_R2',
       'trial_start_pc_timestamp', 'from_ephys_timestamp',
       'to_ephys_timestamp', 'frame_velocity', 'frame_raw',
       'frame_raw_quantile', 'frame_acceleration', 'abs_frame_acceleration',
       'frame_positive_acceleration', 'frame_negative_acceleration',
       'frame_yaw', 'frame_yaw_left', 'frame_yaw_right', 'frame_pitch',
       'frame_pitch_left', 'frame_pitch_right', 'lick_count', 'post_lick',
       'post_reward_sound', 'post_reward', 'frame_position', 'visible_cue',
       'zone_before_reward1', 'zone_before_reward2', 'zone_between_cues',
       'zone_cue2', 'zone_cue2_passed', 'zone_cue2_visible',
       'zone_post_reward', 'zone_reward1', 'zone_reward2', 'head_angle',
       'head_angle_left', 'head_angle_right', 'head_angle_velocity',
       'head_angle_velocity_left', 'head_angle_velocity_right'],
      dtype='object')

In [15]:
behavior_glm_input_flat[chosen_colums].to_csv('behavior_glm_input_flat.csv', index=False)


In [16]:
# drop index level
behavior_glm_input.index = behavior_glm_input.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
drop_cols = ['trial_id',
             'cue',
             'trial_outcome',
             'choice_R1',
             'choice_R2',
             'trial_start_pc_timestamp',
             'from_ephys_timestamp',
             'to_ephys_timestamp',]
behavior_glm_input.drop(drop_cols, axis=1, inplace=True)
# behavior_glm_input.set_index('to_ephys_timestamp', append=True, inplace=True)
behavior_glm_input

,frame_velocity,frame_raw,frame_raw_quantile,frame_acceleration,abs_frame_acceleration,frame_positive_acceleration,frame_negative_acceleration,frame_yaw,frame_yaw_left,frame_yaw_right,...,zone_cue2_visible,zone_post_reward,zone_reward1,zone_reward2,head_angle,head_angle_left,head_angle_right,head_angle_velocity,head_angle_velocity_left,head_angle_velocity_right
session_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
behavior_glm_input[['frame_raw', 'frame_yaw', 'frame_pitch', 'frame_acceleration', 'frame_velocity']].values.shape

(1333016, 5)

In [18]:
behavior_glm_input.index.to_numpy()

array([ 1,  1,  1, ..., 33, 33, 33], shape=(1333016,))

In [19]:
fr.values.shape

(1333016, 77)

In [20]:
import numpy as np
np.save('fr_behavior_glm_input.npy', np.concatenate([behavior_glm_input.index.to_numpy().reshape(-1, 1), fr.values, behavior_glm_input[['frame_raw', 'frame_yaw', 'frame_pitch', 'frame_acceleration', 'frame_velocity']].values], axis=1))

In [39]:
behavior_glm_input = get_and_cache_analytics('Behavior40msAligned', session_names=session_names)
behavior_glm_input.columns
behavior_glm_input_flat = behavior_glm_input.reset_index()
behavior_glm_input_flat.columns

Index(['index', 'paradigm_id', 'animal_id', 'session_id', 'entry_id',
       'trial_id', 'cue', 'trial_outcome', 'choice_R1', 'choice_R2',
       'trial_start_pc_timestamp', 'from_ephys_timestamp',
       'to_ephys_timestamp', 'frame_velocity', 'frame_raw',
       'frame_raw_quantile', 'frame_acceleration', 'abs_frame_acceleration',
       'frame_positive_acceleration', 'frame_negative_acceleration',
       'frame_yaw', 'frame_yaw_left', 'frame_yaw_right', 'frame_pitch',
       'frame_pitch_left', 'frame_pitch_right', 'lick_count', 'post_lick',
       'post_reward_sound', 'post_reward', 'frame_position', 'visible_cue',
       'zone_before_reward1', 'zone_before_reward2', 'zone_between_cues',
       'zone_cue2', 'zone_cue2_passed', 'zone_cue2_visible',
       'zone_post_reward', 'zone_reward1', 'zone_reward2', 'head_angle',
       'head_angle_left', 'head_angle_right', 'head_angle_velocity',
       'head_angle_velocity_left', 'head_angle_velocity_right'],
      dtype='object')

In [38]:
behavior_glm_input_flat[['session_id', 'cue', 'choice_R1', 'choice_R2']].loc[
    behavior_glm_input_flat[['cue', 'choice_R1', 'choice_R2']].notna().any(axis=1)
]

,session_id,cue,choice_R1,choice_R2
85,1,1.0,False,False
86,1,1.0,False,False
87,1,1.0,False,False
88,1,1.0,False,False
89,1,1.0,False,False
...,...,...,...,...
1332172,33,2.0,False,True
1332173,33,2.0,False,True
1332174,33,2.0,False,True
1332175,33,2.0,False,True


In [33]:
behavior_glm_input_flat.columns


Index(['index', 'paradigm_id', 'animal_id', 'session_id', 'entry_id',
       'trial_id', 'cue', 'trial_outcome', 'choice_R1', 'choice_R2',
       'trial_start_pc_timestamp', 'from_ephys_timestamp',
       'to_ephys_timestamp', 'frame_velocity', 'frame_raw',
       'frame_raw_quantile', 'frame_acceleration', 'abs_frame_acceleration',
       'frame_positive_acceleration', 'frame_negative_acceleration',
       'frame_yaw', 'frame_yaw_left', 'frame_yaw_right', 'frame_pitch',
       'frame_pitch_left', 'frame_pitch_right', 'lick_count', 'post_lick',
       'post_reward_sound', 'post_reward', 'frame_position', 'visible_cue',
       'zone_before_reward1', 'zone_before_reward2', 'zone_between_cues',
       'zone_cue2', 'zone_cue2_passed', 'zone_cue2_visible',
       'zone_post_reward', 'zone_reward1', 'zone_reward2', 'head_angle',
       'head_angle_left', 'head_angle_right', 'head_angle_velocity',
       'head_angle_velocity_left', 'head_angle_velocity_right'],
      dtype='object')

In [69]:
behavior_glm_input_flat[['session_id', 'visible_cue', 'cue']].loc[
    behavior_glm_input_flat['visible_cue']!= 0.0
].head(200)

,session_id,visible_cue,cue
47,1,1.0,NaN
48,1,1.0,NaN
49,1,1.0,NaN
50,1,1.0,NaN
51,1,1.0,NaN
...,...,...,...
946,1,1.0,1.0
947,1,1.0,1.0
948,1,1.0,1.0
949,1,1.0,1.0
